# Lab12 - YOLOv8 物件偵測


## 1. 安裝 Ultralytics 套件


In [ ]:
%pip install -U ultralytics

## 2. Import 必要套件


In [ ]:
import os
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

import torch
import yaml
from IPython.display import Image, display
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

from urllib.request import urlretrieve
from PIL import Image as PILImage


## 3. VOC2007 資料集設定

Ultralytics 使用 YOLO 格式標註：每張圖片對應一個 `.txt`，每列為
`class cx cy w h`，座標皆為 0 到 1 的 normalized 值。下方 cell 會把 VOC XML
標註轉成 YOLO 格式，並建立 `voc2007_yolo/data.yaml`。


In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}

PROJECT_ROOT = Path.cwd()
TRAIN_ROOT = PROJECT_ROOT / "VOCtrainval_06-Nov-2007" / "VOCdevkit" / "VOC2007"
TEST_ROOT = PROJECT_ROOT / "VOCtest_06-Nov-2007" / "VOCdevkit" / "VOC2007"
YOLO_DATA_ROOT = PROJECT_ROOT / "voc2007_yolo"

RUN_NAME = "YOLOv8_VOC2007_ultralytics"
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"
IMAGE_SIZE = 416
# 依 GPU VRAM 調整 batch size；依收斂狀況調整 epochs。
BATCH_SIZE = 128
EPOCHS = 30
MODEL_WEIGHTS = "yolov8n.pt"
WORKERS = 0
SEED = 42

random.seed(SEED)
SETTINGS.update({"tensorboard": True})  # 啟用 TensorBoard event log。


print("train root:", TRAIN_ROOT)
print("test root:", TEST_ROOT)
print("classes:", len(VOC_CLASSES))

In [ ]:
# YOLO 標籤需要把 bbox 轉成相對座標，中心點與寬高都正規化到 0~1。
def voc_box_to_yolo(size, box):
    width, height = size
    xmin, ymin, xmax, ymax = box
    cx = ((xmin + xmax) / 2) / width
    cy = ((ymin + ymax) / 2) / height
    bw = (xmax - xmin) / width
    bh = (ymax - ymin) / height
    return cx, cy, bw, bh


def convert_voc_split(voc_root, image_set, split_name):
    image_ids_path = voc_root / "ImageSets" / "Main" / f"{image_set}.txt"
    image_ids = [line.strip() for line in image_ids_path.read_text().splitlines() if line.strip()]

    # Ultralytics 預期 images/ 與 labels/ 依 split 對齊，同名 txt 存放每張圖的框。
    image_out = YOLO_DATA_ROOT / "images" / split_name
    label_out = YOLO_DATA_ROOT / "labels" / split_name
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)

    used = 0
    for image_id in image_ids:
        xml_path = voc_root / "Annotations" / f"{image_id}.xml"
        image_path = voc_root / "JPEGImages" / f"{image_id}.jpg"
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find("size")
        width = int(size.findtext("width"))
        height = int(size.findtext("height"))

        labels = []
        for obj in root.findall("object"):
            difficult = int(obj.findtext("difficult", default="0"))
            class_name = obj.findtext("name")
            # VOC difficult 樣本不納入訓練，避免標註不清楚的目標干擾學習。
            if difficult == 1 or class_name not in CLASS_TO_ID:
                continue

            # VOC bbox: xmin, ymin, xmax, ymax -> YOLO: cx, cy, w, h.
            bndbox = obj.find("bndbox")
            xmin = float(bndbox.findtext("xmin"))
            ymin = float(bndbox.findtext("ymin"))
            xmax = float(bndbox.findtext("xmax"))
            ymax = float(bndbox.findtext("ymax"))
            cx, cy, bw, bh = voc_box_to_yolo((width, height), (xmin, ymin, xmax, ymax))
            labels.append(f"{CLASS_TO_ID[class_name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # 即使沒有可用目標也保留空標籤檔，讓圖片與標籤數量維持一致。
        shutil.copy2(image_path, image_out / image_path.name)
        (label_out / f"{image_id}.txt").write_text("\n".join(labels))
        used += 1

    return used


train_count = convert_voc_split(TRAIN_ROOT, "trainval", "train")
val_count = convert_voc_split(TEST_ROOT, "test", "val")

# data.yaml 告訴 Ultralytics 訓練/驗證資料與類別名稱位置。
data_yaml = YOLO_DATA_ROOT / "data.yaml"
data = {
    "path": str(YOLO_DATA_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {idx: name for idx, name in enumerate(VOC_CLASSES)},
}
data_yaml.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))

print("converted train images:", train_count)
print("converted val images:", val_count)
print("data yaml:", data_yaml)


## 4. 影像增強（Image Augmentation）

Ultralytics YOLO 訓練時預設會使用多種影像增強。mosaic 會將多張訓練影像透過隨機縮放與隨機位置拼接成新的訓練樣本，讓模型學習不同尺度與背景組合。下方 cell 示範不含標註框的 mosaic 拼接效果。


In [ ]:
mosaic_size = IMAGE_SIZE
sample_count = 4
image_pool = sorted((YOLO_DATA_ROOT / "images" / "train").glob("*.jpg"))
sample_images = random.sample(image_pool, min(sample_count, len(image_pool)))

mosaic = PILImage.new("RGB", (mosaic_size, mosaic_size), color=(114, 114, 114))

for image_path in sample_images:
    image = PILImage.open(image_path).convert("RGB")
    src_w, src_h = image.size

    # 隨機縮放後貼到隨機位置，超出畫布的部分會裁掉。
    scale = random.uniform(0.35, 0.8)
    target_long_side = int(mosaic_size * scale)
    resize_ratio = target_long_side / max(src_w, src_h)
    new_w = max(1, int(src_w * resize_ratio))
    new_h = max(1, int(src_h * resize_ratio))
    image = image.resize((new_w, new_h))

    max_x = mosaic_size - new_w
    max_y = mosaic_size - new_h
    paste_x = random.randint(min(0, max_x), max(0, max_x))
    paste_y = random.randint(min(0, max_y), max(0, max_y))

    # 負座標代表圖片貼出畫布外，需要先裁切再貼回可視範圍。
    crop_left = max(0, -paste_x)
    crop_top = max(0, -paste_y)
    crop_right = min(new_w, mosaic_size - paste_x)
    crop_bottom = min(new_h, mosaic_size - paste_y)

    if crop_right <= crop_left or crop_bottom <= crop_top:
        continue

    tile = image.crop((crop_left, crop_top, crop_right, crop_bottom))
    mosaic.paste(tile, (max(0, paste_x), max(0, paste_y)))

display(mosaic)


## 5. 下載預訓練權重並建立模型

下載 YOLOv8n 預訓練權重 `yolov8n.pt`


In [ ]:

MODEL_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt"
weights_path = PROJECT_ROOT / MODEL_WEIGHTS

# 若本地已有權重就直接使用，避免重複下載。
if MODEL_WEIGHTS.endswith(".pt"):
    if not weights_path.exists():
        print(f"Downloading {MODEL_WEIGHTS} from {MODEL_URL}")
        urlretrieve(MODEL_URL, weights_path)
    else:
        print(f"Found existing weights: {weights_path}")
    MODEL_SOURCE = str(weights_path)
    print(f"weights size: {weights_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    MODEL_SOURCE = MODEL_WEIGHTS
    print(f"Using model config: {MODEL_SOURCE}")


建立模型


In [ ]:
model = YOLO(MODEL_SOURCE)
model.info()

## 6. 訓練與驗證網路模型

Ultralytics 會處理 dataloader、augmentation、loss、checkpoint 與 TensorBoard 紀錄。訓練結果會放在 `runs/detect/YOLOv8_VOC2007_ultralytics/`。


In [ ]:
train_results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    pretrained=True,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
)


使用訓練後的 `best.pt` 在 VOC2007 test split 上做驗證。


In [ ]:
# 載入訓練過程中驗證表現最佳的權重。
best_model_path = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

# 使用同一份 data.yaml 驗證，確保類別順序與訓練時完全一致。
metrics = best_model.val(
    data=str(data_yaml),
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=0 if torch.cuda.is_available() else "cpu",
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)

## 8. 推論結果

從 VOC2007 test split 抽 16 張圖片做物件偵測，輸出到 `runs/detect/YOLOv8_VOC2007_ultralytics_predict/`。


In [ ]:
val_images = sorted((YOLO_DATA_ROOT / "images" / "val").glob("*.jpg"))
sample_images = random.sample(val_images, k=min(16, len(val_images)))

# conf 控制最低信心分數，iou 控制 NMS 合併重疊框的嚴格程度。
pred_results = best_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.5,
    save=True,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_predict",
    device=0 if torch.cuda.is_available() else "cpu",
)

predict_dir = RUNS_DIR / f"{RUN_NAME}_predict"
print("prediction dir:", predict_dir)

for image_path in sorted(predict_dir.glob("*.jpg"))[:8]:
    display(Image(filename=str(image_path), width=420))


## 9. TensorBoard

訓練前已啟用 Ultralytics TensorBoard callback，訓練完成後可以使用TensorBoard查看訓練曲線。


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/detect
